# StadtRad Hamburg — trip inference & visualization

Reads the per-bike snapshots written by `scripts/gbfs_poller.py` and infers
rentals from the gap between a bike being last seen at one station and next
seen at another. Produces four artifacts in `analysis/out/`:

1. `trips_timeseries.png` — rentals per 15-min bucket
2. `top_stations.png`     — top-20 origins & destinations
3. `trip_durations.png`   — inferred duration histogram
4. `trips_map.html`       — interactive Folium OD map

Trip-time bounds are ± one poll interval at each endpoint, so duration
estimates are coarse. Sub-interval and same-station-return trips are
invisible by construction.

In [ ]:
from pathlib import Path
import duckdb, folium
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'analysis':
    REPO_ROOT = REPO_ROOT.parent
DATA = REPO_ROOT / 'data' / 'gbfs'
OUT  = REPO_ROOT / 'analysis' / 'out'
OUT.mkdir(parents=True, exist_ok=True)

BIKES_GLOB    = str(DATA / 'bikes' / 'date=*' / 'part-*.parquet')
STATIONS_FILE = str(DATA / 'stations.parquet')

con = duckdb.connect()
con.execute(f"CREATE VIEW bikes    AS SELECT * FROM '{BIKES_GLOB}'")
con.execute(f"CREATE VIEW stations AS SELECT * FROM '{STATIONS_FILE}'")

summary = con.execute("""
  SELECT COUNT(*)                   AS rows,
         COUNT(DISTINCT bike_number) AS unique_bikes,
         MIN(observed_at)            AS first_seen,
         MAX(observed_at)            AS last_seen
  FROM bikes
""").fetchone()
print(f"bike rows:    {summary[0]:>10,}")
print(f"unique bikes: {summary[1]:>10,}")
print(f"span (UTC):   {summary[2]} → {summary[3]}")
print(f"stations:     {con.execute('SELECT COUNT(*) FROM stations').fetchone()[0]:>10,}")

## Trip inference

For each bike, take consecutive snapshots ordered by `observed_at`. A change in
`station_id` between adjacent snapshots is a completed rental. We discard
gaps > 6 h (these are typically bikes that went offline / were withdrawn for
maintenance rather than a single ride).

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE trips AS
WITH ordered AS (
  SELECT bike_number, station_id, observed_at,
         LAG(station_id)  OVER w AS prev_station,
         LAG(observed_at) OVER w AS prev_seen
  FROM bikes
  WHERE station_id IS NOT NULL
  WINDOW w AS (PARTITION BY bike_number ORDER BY observed_at)
)
SELECT bike_number,
       prev_station AS origin_id,
       station_id   AS destination_id,
       prev_seen    AS departed_after,
       observed_at  AS arrived_by,
       observed_at - prev_seen AS gap
FROM ordered
WHERE prev_station IS NOT NULL
  AND prev_station <> station_id
  AND observed_at - prev_seen < INTERVAL 6 HOUR
""")

con.execute("""
CREATE OR REPLACE TABLE trips_geo AS
SELECT t.*,
       o.name AS origin_name, o.lat AS origin_lat, o.lon AS origin_lon,
       d.name AS dest_name,   d.lat AS dest_lat,   d.lon AS dest_lon
FROM trips t
JOIN stations o ON o.station_id = t.origin_id
JOIN stations d ON d.station_id = t.destination_id
""")

stats = con.execute("""
  SELECT COUNT(*) AS trips,
         COUNT(DISTINCT bike_number) AS bikes_that_moved,
         median(epoch(gap) / 60) AS median_gap_min,
         min(gap) AS min_gap,
         max(gap) AS max_gap
  FROM trips_geo
""").fetchone()
print(f"trips inferred:      {stats[0]:>8,}")
print(f"distinct bikes used: {stats[1]:>8,}")
print(f"median gap (min):    {stats[2]:>8.1f}")
print(f"min / max gap:       {stats[3]} / {stats[4]}")

## Plot 1 — trips per 15 minutes (UTC)

In [ ]:
df_ts = con.execute("""
  SELECT time_bucket(INTERVAL 15 MINUTE, arrived_by) AS bucket,
         COUNT(*) AS trips
  FROM trips_geo
  GROUP BY 1
  ORDER BY 1
""").df()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(df_ts['bucket'], df_ts['trips'], marker='o', markersize=3, linewidth=1)
ax.set_title('StadtRad Hamburg — inferred rentals per 15 min (UTC)')
ax.set_xlabel('arrival bucket (UTC)')
ax.set_ylabel('trips')
ax.grid(alpha=0.3)
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(OUT / 'trips_timeseries.png', dpi=140)
plt.show()

## Plot 2 — top origins & destinations

In [ ]:
top_origin = con.execute("""
  SELECT origin_name AS name, COUNT(*) AS n
  FROM trips_geo GROUP BY 1 ORDER BY n DESC LIMIT 20
""").df()
top_dest = con.execute("""
  SELECT dest_name AS name, COUNT(*) AS n
  FROM trips_geo GROUP BY 1 ORDER BY n DESC LIMIT 20
""").df()

fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 7), sharex=False)
axL.barh(top_origin['name'][::-1], top_origin['n'][::-1], color='steelblue')
axL.set_title('Top 20 origins (departures)')
axL.set_xlabel('trips')
axR.barh(top_dest['name'][::-1], top_dest['n'][::-1], color='darkorange')
axR.set_title('Top 20 destinations (arrivals)')
axR.set_xlabel('trips')
for ax in (axL, axR):
    ax.grid(axis='x', alpha=0.3)
fig.tight_layout()
fig.savefig(OUT / 'top_stations.png', dpi=140)
plt.show()

## Plot 3 — trip-duration histogram

Bins of 5 minutes. The lowest bin is constrained from below by the poll
interval (5 min), so trips faster than that show up as the leftmost bin or are
invisible entirely. Duration error per trip is roughly ±5 min.

In [ ]:
gap_min = con.execute("SELECT epoch(gap) / 60 AS minutes FROM trips_geo").df()['minutes']

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(gap_min, bins=range(0, 125, 5), color='seagreen', edgecolor='white')
ax.set_title('Inferred trip duration distribution')
ax.set_xlabel('minutes (between last-seen-at-origin and first-seen-at-destination)')
ax.set_ylabel('trips')
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig(OUT / 'trip_durations.png', dpi=140)
plt.show()

print(f"median: {gap_min.median():.1f} min   p90: {gap_min.quantile(0.9):.1f} min   max kept: {gap_min.max():.1f} min")

## Plot 4 — interactive OD map

Stations sized by total activity (departures + arrivals). The 50 heaviest OD
pairs are drawn as straight lines — these are *not* routes, just OD endpoints,
since GBFS does not expose path geometry.

In [ ]:
station_activity = con.execute("""
  SELECT name, lat, lon, departures + arrivals AS total
  FROM (
    SELECT s.station_id, s.name, s.lat, s.lon,
           COALESCE(dep.n, 0) AS departures,
           COALESCE(arr.n, 0) AS arrivals
    FROM stations s
    LEFT JOIN (SELECT origin_id AS sid, COUNT(*) AS n FROM trips_geo GROUP BY 1) dep ON dep.sid = s.station_id
    LEFT JOIN (SELECT destination_id AS sid, COUNT(*) AS n FROM trips_geo GROUP BY 1) arr ON arr.sid = s.station_id
  )
  WHERE total > 0
""").df()

top_od = con.execute("""
  SELECT origin_lat, origin_lon, dest_lat, dest_lon,
         origin_name, dest_name, COUNT(*) AS n
  FROM trips_geo
  GROUP BY 1,2,3,4,5,6
  ORDER BY n DESC
  LIMIT 50
""").df()

m = folium.Map(location=[53.55, 10.00], zoom_start=12, tiles='cartodbpositron')

max_total = max(station_activity['total'].max(), 1)
for row in station_activity.itertuples(index=False):
    folium.CircleMarker(
        location=[row.lat, row.lon],
        radius=2 + 14 * (row.total / max_total) ** 0.5,
        color='#225599', weight=1, fill=True, fill_opacity=0.55,
        tooltip=f"{row.name}: {row.total} trips"
    ).add_to(m)

max_od = max(top_od['n'].max(), 1)
for r in top_od.itertuples(index=False):
    folium.PolyLine(
        locations=[[r.origin_lat, r.origin_lon], [r.dest_lat, r.dest_lon]],
        color='#c0392b',
        weight=1 + 5 * (r.n / max_od),
        opacity=0.55,
        tooltip=f"{r.origin_name} → {r.dest_name}: {r.n} trips"
    ).add_to(m)

map_path = OUT / 'trips_map.html'
m.save(str(map_path))
print(f"map written to {map_path}")
m